In [ ]:
# base
import pandas as pd
import gc
import warnings

# models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# optimization hyperparameters
import optuna
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import root_mean_squared_error

In [24]:
warnings.filterwarnings('ignore')

df = pd.read_csv("data/train.csv")
df

,exper,expersq,black,south66,enroll,smsa,momdad14,weight,wage,libcrd14,motheduc,fatheduc,KWW,IQ,educ
0,16,256,1,0,0,1,1,158413,548,0.0,10.348137,10.003448,15.0,102.449782,7
1,9,81,0,0,0,1,1,380166,481,1.0,8.000000,8.000000,35.0,93.000000,12
2,16,256,0,0,0,1,1,367470,721,1.0,12.000000,14.000000,42.0,103.000000,12
3,10,100,0,0,0,1,1,380166,250,1.0,12.000000,11.000000,25.0,88.000000,11
4,16,256,0,0,0,1,1,367470,729,0.0,7.000000,8.000000,34.0,108.000000,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3005,7,49,0,1,0,0,1,82135,335,0.0,12.000000,8.000000,15.0,102.449782,12
3006,15,225,0,1,0,1,1,88765,481,1.0,10.348137,10.003448,43.0,102.449782,13
3007,6,36,0,1,0,0,0,89271,500,0.0,10.348137,11.000000,25.0,109.000000,12
3008,13,169,0,1,0,0,1,110376,713,1.0,10.348137,10.003448,32.0,107.000000,12


In [25]:
def run_hyperopt(name, obj_fn, n_trials=50):
    study = optuna.create_study(
        direction="minimize",
        study_name=f"{name}_drone_energy",
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=20),
    )

    study.optimize(
        lambda trial: obj_fn(trial, X, y),
        n_trials=n_trials,
    )
    return study

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

MODEL_REGISTRY = {
    "ridge": {
        "model_cls": Ridge,
        "param_space": lambda trial: {
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        },
    },
    "lasso": {
        "model_cls": Lasso,
        "param_space": lambda trial: {
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        },
    },
    "rf": {
        "model_cls": RandomForestRegressor,
        "param_space": lambda trial: {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "random_state": 42,
            "n_jobs": -1,
        },
    },
}


def make_objective(model_key, X, y, cv=kf, groups=None):
    """Factory: returns an Optuna objective for the given model key."""
    spec = MODEL_REGISTRY[model_key]

    def objective(trial):
        params = spec["param_space"](trial)
        model = spec["model_cls"](**params)

        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            groups=groups,  # only used if cv is a grouped splitter
            scoring="neg_root_mean_squared_error",
            n_jobs=-1,
        )
        return -scores.mean()  # convert back to positive RMSE

    return objective

In [28]:
# feature_cols = ['exper', 'expersq', 'black', 'south66', 'enroll', 'smsa',
#                 'momdad14', 'weight', 'wage', 'libcrd14', 'motheduc', 'fatheduc',
#                 'KWW', 'IQ']
target = 'educ'
X = df.drop(columns=[target]).values
y = df[target].values

def run_hyperopt(name, X, y, n_trials=50, cv=kf, groups=None):
    study = optuna.create_study(direction="minimize", study_name=name)
    study.optimize(
        make_objective(name, X, y, cv=cv, groups=groups),
        n_trials=n_trials,
    )
    return study

studies = {}
n_trials_map = {"ridge": 50, "lasso": 50, "rf": 100}

for name in MODEL_REGISTRY:
    study = run_hyperopt(name, X, y, n_trials=n_trials_map[name])
    studies[name] = study
    print(f"{name} best RMSE: {study.best_value:.4f} | params: {study.best_params}")

[I 2026-07-19 13:25:26,719] A new study created in memory with name: ridge
[I 2026-07-19 13:25:26,734] Trial 0 finished with value: 1.522800845859026 and parameters: {'alpha': 0.1800842471536599}. Best is trial 0 with value: 1.522800845859026.
[I 2026-07-19 13:25:26,749] Trial 1 finished with value: 1.5227852240697475 and parameters: {'alpha': 1.9095827460072894}. Best is trial 1 with value: 1.5227852240697475.
[I 2026-07-19 13:25:26,764] Trial 2 finished with value: 1.5228026089305327 and parameters: {'alpha': 0.007156277077274548}. Best is trial 1 with value: 1.5227852240697475.
[I 2026-07-19 13:25:26,778] Trial 3 finished with value: 1.5228026430736041 and parameters: {'alpha': 0.0038430448422226908}. Best is trial 1 with value: 1.5227852240697475.
[I 2026-07-19 13:25:26,793] Trial 4 finished with value: 1.5227956389756403 and parameters: {'alpha': 0.713644928903184}. Best is trial 1 with value: 1.5227852240697475.
[I 2026-07-19 13:25:26,807] Trial 5 finished with value: 1.522802662

ridge best RMSE: 1.5228 | params: {'alpha': 9.45428340890265}


[I 2026-07-19 13:25:27,682] Trial 12 finished with value: 1.5228119861031932 and parameters: {'alpha': 0.0014238259735528632}. Best is trial 10 with value: 1.5228042457919286.
[I 2026-07-19 13:25:27,697] Trial 13 finished with value: 1.5228072874906 and parameters: {'alpha': 0.001216492897047834}. Best is trial 10 with value: 1.5228042457919286.
[I 2026-07-19 13:25:27,712] Trial 14 finished with value: 1.5232884906088011 and parameters: {'alpha': 0.005999999957384242}. Best is trial 10 with value: 1.5228042457919286.
[I 2026-07-19 13:25:27,728] Trial 15 finished with value: 1.5229339446148897 and parameters: {'alpha': 0.0034337966281929766}. Best is trial 10 with value: 1.5228042457919286.
[I 2026-07-19 13:25:27,753] Trial 16 finished with value: 1.5228908477762855 and parameters: {'alpha': 0.0029361700720761702}. Best is trial 10 with value: 1.5228042457919286.
[I 2026-07-19 13:25:27,768] Trial 17 finished with value: 1.522804179648003 and parameters: {'alpha': 0.0010188701430208847}.

lasso best RMSE: 1.5228 | params: {'alpha': 0.0010074377040584855}


[I 2026-07-19 13:25:28,662] Trial 0 finished with value: 1.507749834766582 and parameters: {'n_estimators': 223, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 1.507749834766582.
[I 2026-07-19 13:25:29,171] Trial 1 finished with value: 1.5041637259942937 and parameters: {'n_estimators': 371, 'max_depth': 6, 'min_samples_split': 3, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 1.5041637259942937.
[I 2026-07-19 13:25:29,681] Trial 2 finished with value: 1.931714851627504 and parameters: {'n_estimators': 381, 'max_depth': 2, 'min_samples_split': 3, 'min_samples_leaf': 9, 'max_features': None}. Best is trial 1 with value: 1.5041637259942937.
[I 2026-07-19 13:25:30,248] Trial 3 finished with value: 1.4128822906806398 and parameters: {'n_estimators': 275, 'max_depth': 7, 'min_samples_split': 15, 'min_samples_leaf': 9, 'max_features': None}. Best is trial 3 with value: 1.4128822906806398.
[I 202

rf best RMSE: 1.3697 | params: {'n_estimators': 412, 'max_depth': 8, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None}


In [22]:
del df
gc.collect()
%whos

Variable                  Type        Data/Info
-----------------------------------------------
KFold                     ABCMeta     <class 'sklearn.model_selection._split.KFold'>
Lasso                     ABCMeta     <class 'sklearn.linear_mo<...>oordinate_descent.Lasso'>
RandomForestRegressor     ABCMeta     <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
Ridge                     ABCMeta     <class 'sklearn.linear_model._ridge.Ridge'>
X                         ndarray     3010x15: 45150 elems, type `float64`, 361200 bytes (352.734375 kb)
cross_val_score           function    <function cross_val_score at 0x7f8be4884e00>
feature_cols              list        n=14
gc                        module      <module 'gc' (built-in)>
kf                        KFold       KFold(n_splits=5, random_state=42, shuffle=True)
name                      str         rf
np                        module      <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
obj_fn                    functio